In [1]:
# Our architecture relies on a heterogeneous gradient boosting ensemble. We import
# standard data manipulation libraries (pandas, numpy) alongside our two chosen 
# modeling frameworks: CatBoost (selected for its Ordered Target Encoding and 
# symmetric tree regularization) and LightGBM (selected for its asymmetric, 
# leaf-wise tree growth). We suppress repetitive console warnings to maintain 
# a clean standard output during the massive 12,000-iteration training loops.
# ==============================================================================
import warnings
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
import lightgbm as lgb

warnings.filterwarnings('ignore')

In [2]:
# The core problem with this dataset is a severe temporal distribution shift. 
# The training data only covers Days 48 and 49, leaving the model structurally 
# blind to the remaining hidden days in the test set. To solve this, we load 
# both the raw datasets and our highest-scoring prior submission, which will 
# serve as the foundation for our semi-supervised pseudo-labeling pipeline.
# ==============================================================================
train = pd.read_csv('train.csv')
test  = pd.read_csv('test.csv')
best_sub = pd.read_csv('submission_FINAL_ENSEMBLE.csv')

In [3]:
# We append our high-confidence baseline predictions to the test set as if they 
# were ground truth. By concatenating this artificially labeled test data with 
# the original training data, we create a unified master corpus. This exposes 
# the upcoming models to the actual temporal distributions, geographical boundaries, 
# and weather patterns of the hidden month, allowing the ensemble to dynamically 
# correct its own micro-errors across the spatial grid.
# ==============================================================================
test_pseudo = test.copy()
test_pseudo['demand'] = best_sub['demand']

combined_data = pd.concat([train, test_pseudo], axis=0).reset_index(drop=True)

In [4]:
# We intentionally omit continuous coordinates (latitude/longitude) to prevent 
# the "Overfitting Paradox," where trees perfectly memorize the traffic of Day 48.
# Instead, we build a time-invariant feature space where the mathematical logic 
# holds true regardless of the calendar day.
# ==============================================================================
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    
    # 1. Temporal Decomposition: Extract numeric components from the timestamp string.
    parts = df['timestamp'].str.split(':', expand=True).astype(int)
    df['hour'] = parts[0]
    df['minute'] = parts[1]
    
    # Map the 24-hour day into a continuous index of 96 discrete 15-minute intervals.
    df['time_slot'] = df['hour'] * 4 + df['minute'] // 15
    
    # 2. Cyclical Time Transformation: Linear time causes a structural boundary issue 
    # (23:45 is mathematically far from 00:00). We apply trigonometric transformations 
    # to project time onto a circle, ensuring continuous temporal proximity.
    df['sin_time'] = np.sin(2 * np.pi * df['time_slot'] / 96)
    df['cos_time'] = np.cos(2 * np.pi * df['time_slot'] / 96)
    df['time_slot_cat'] = 'Slot_' + df['time_slot'].astype(str)
    
    # 3. Unseen Category Regularization: Retaining 'day_of_week' forces the model to 
    # use global neighborhood averages when it encounters an unseen day (like Sunday) 
    # in the test set, rather than hallucinating known patterns from a Tuesday.
    df['day_of_week'] = (df['day'] % 7).astype(str)
    df['is_weekend'] = df['day_of_week'].isin(['5', '6']).astype(str)
    
    # 4. Spatial Hierarchies: Full 6-character geohashes are highly sparse. We extract 
    # 5-character and 4-character substrings to act as structural district fallbacks. 
    # If a specific intersection is unseen, the model safely defaults to the parent region.
    df['geohash_5'] = df['geohash'].str[:5]
    df['geohash_4'] = df['geohash'].str[:4]
    
    # 5. The Trigger Feature: By crossing the geohash string with the time slot category,
    # we create an interaction feature. Passing this to CatBoost triggers its internal 
    # Ordered Target Encoding (OTE), extracting historical baselines without data leakage.
    df['geo_time'] = df['geohash'].astype(str) + "_" + df['time_slot_cat'].astype(str)
    
    return df

In [5]:
# We apply the stateless feature engineering function identically to both the 
# unified training corpus and the pure test set. This strict parity ensures that 
# the models encounter the exact same feature dimensionalities during inference.
# ==============================================================================
combined_data = build_features(combined_data)
test          = build_features(test)

In [6]:
# Gradient boosting frameworks handle categoricals differently. LightGBM relies 
# on pandas 'category' dtypes for optimal memory mapping and histogram building. 
# We standardize missing environmental data using out-of-range sentinel values (-999) 
# so the decision trees can safely isolate null occurrences into their own nodes.
# ==============================================================================
combined_data['Temperature'] = combined_data['Temperature'].fillna(-999)
test['Temperature']          = test['Temperature'].fillna(-999)

CAT_FEATURES = [
    'geohash', 'geohash_5', 'geohash_4', 'RoadType', 'LargeVehicles', 
    'Landmarks', 'Weather', 'time_slot_cat', 'day_of_week', 'is_weekend',
    'geo_time'
]

# Cast to explicitly mapped pandas categorical types.
for col in CAT_FEATURES:
    combined_data[col] = combined_data[col].fillna('Unknown').astype('category')
    test[col]          = test[col].fillna('Unknown').astype('category')

FEATURES = [
    'geohash', 'geohash_5', 'geohash_4', 'hour', 'minute', 'time_slot', 'time_slot_cat',
    'day_of_week', 'is_weekend', 'geo_time', 'RoadType', 'NumberofLanes', 
    'LargeVehicles', 'Landmarks', 'Temperature', 'Weather', 'sin_time', 'cos_time'
]
TARGET = 'demand'

In [7]:
# The traffic demand target exhibits a heavily right-skewed, long-tail distribution 
# with massive bottleneck spikes. Because RMSE disproportionately punishes these 
# extreme outliers, we apply a log1p transformation. This mathematically squashes 
# the anomalies, forcing the trees to converge on subtle baseline traffic variations 
# rather than wasting depth isolating single traffic jams.
# ==============================================================================
combined_data['log_demand'] = np.log1p(combined_data[TARGET])

In [ ]:
# We initialize a multi-seed training loop. Our primary anchor is CatBoost, 
# which receives a 60% blend weight. Our secondary explorer is LightGBM, 
# receiving a 40% blend weight. Both models run inside this unified loop.
# ==============================================================================
SEEDS = [42, 777, 2024]
test_preds_log = np.zeros(len(test))

for seed in SEEDS:
    # --- MODEL A: CATBOOST ---
    cb_model = CatBoostRegressor(
        iterations=12000,           
        learning_rate=0.015,        
        depth=8, 
        l2_leaf_reg=6,              
        subsample=0.80,
        bootstrap_type='Bernoulli', 
        loss_function='RMSE', 
        random_seed=seed, 
        verbose=0
    )
    
    cb_cat_data = combined_data.copy()
    cb_test_data = test.copy()
    for col in CAT_FEATURES:
        cb_cat_data[col] = cb_cat_data[col].astype(str)
        cb_test_data[col] = cb_test_data[col].astype(str)
        
    cb_model.fit(cb_cat_data[FEATURES], cb_cat_data['log_demand'], cat_features=CAT_FEATURES)
    cb_preds = cb_model.predict(cb_test_data[FEATURES])
    
    # --- MODEL B: LIGHTGBM ---
    lgb_model = lgb.LGBMRegressor(
        n_estimators=12000,
        learning_rate=0.01,         
        num_leaves=63,
        subsample=0.80,
        colsample_bytree=0.80,
        reg_alpha=0.1,              
        reg_lambda=5.0,             
        random_state=seed,
        n_jobs=-1,
        verbose=-1
    )
    
    lgb_model.fit(combined_data[FEATURES], combined_data['log_demand'], categorical_feature=CAT_FEATURES)
    lgb_preds = lgb_model.predict(test[FEATURES])
    
    # --- BLEND & ACCUMULATE ---
    blended_seed_preds = (cb_preds * 0.6) + (lgb_preds * 0.4)
    test_preds_log += blended_seed_preds / len(SEEDS)

In [ ]:
# We revert the aggregated predictions from log-space back to their true spatial scale. 
# However, reversing a logarithmic mean introduces a downward arithmetic bias known 
# as Jensen's Inequality (E[f(X)] >= f(E[X]) for concave functions). To correct 
# this, we apply a tuned scalar multiplier of 1.005. We then clip the bounds to 
# respect physical constraints (traffic cannot be negative) and export the final CSV.
# ==============================================================================
final_test_preds = np.expm1(test_preds_log)

# Apply Jensen's Inequality correction factor
final_test_preds = final_test_preds * 1.005 

# Enforce real-world physical boundaries
final_test_preds = np.clip(final_test_preds, 0, None)

submission = pd.DataFrame({'Index': test['Index'], 'demand': final_test_preds})
submission.to_csv('submission_FINAL_DEEP.csv', index=False)